# LlamaIndex와 AgentCore 메모리 - 학술 연구 어시스턴트 (장기 메모리)

## 소개

이 노트북은 Amazon Bedrock AgentCore 메모리 기능을 LlamaIndex와 통합하여 여러 연구 세션에 걸쳐 **장기 메모리** 지속성을 가진 학술 연구 어시스턴트를 만드는 방법을 보여줍니다. 수주 및 수개월에 걸친 연구 작업에서 누적 지식을 구축할 수 있습니다.

## 아키텍처 개요

![LlamaIndex AgentCore 장기 메모리 아키텍처](LlamaIndex-AgentCore-LTM-Arch.png)

## 튜토리얼 세부 정보

**튜토리얼 세부 정보:**
- **튜토리얼 유형**: 장기 크로스 세션 메모리
- **에이전트 사용 사례**: 학술 연구 어시스턴트
- **에이전트 프레임워크**: LlamaIndex
- **LLM 모델**: Anthropic Claude 3.7 Sonnet
- **튜토리얼 구성 요소**: AgentCore 장기 메모리, LlamaIndex 에이전트, 연구 도구
- **예제 난이도**: 고급

## 비즈니스 가치

**엔터프라이즈 연구 인텔리전스**: 조직 지식을 축적하고, 연구 발전을 추적하며, 프로젝트와 기간에 걸쳐 포괄적인 학술 컨텍스트를 유지하는 영구적 AI 메모리로 연구 워크플로를 혁신합니다.

**주요 전문적 장점:**
- **연구 연속성**: 연구 단계와 팀원 간의 원활한 지식 이전
- **조직 메모리**: 중요한 연구 인사이트, 방법론 및 결과를 영구적으로 보존
- **크로스 프로젝트 인텔리전스**: 여러 연구 이니셔티브에 걸친 패턴과 연결 식별
- **연구비 제안서 우수성**: 설득력 있는 자금 신청을 위한 과거 연구 데이터 활용
- **학술 협력**: 다년간의 공동 연구 프로젝트를 위한 상세한 컨텍스트 유지
- **출판 전략**: 전략적 출판 계획을 위한 연구 주제와 인용 네트워크 추적

## 장기 메모리 구성

**기술 설정**: 이 튜토리얼은 12개월 보존을 위한 시맨틱 전략과 함께 AgentCore 메모리를 사용합니다:
- **메모리 유형**: 자동 인사이트 추출이 가능한 시맨틱 전략
- **보존 기간**: 연구 연속성을 위한 365일 이벤트 만료
- **크로스 세션**: 동일한 actor_id + memory_id, 연구 기간별 다른 session_id
- **검색 기능**: 연구 이력에 걸친 시맨틱 검색을 위한 내장 메모리 검색 도구

## 기술 개요

**주요 장기 메모리 구성 요소:**
1. **시맨틱 전략 구성**: 365일 보존으로 자동 인사이트 추출을 위한 SemanticStrategy 사용
2. **크로스 세션 지속성**: 동일한 actor_id + memory_id, 기간별 다른 session_id로 지식 연속성 실현
3. **커스텀 메모리 검색 도구**: AgentCore의 네이티브 search_long_term_memories()를 LlamaIndex FunctionTool로 래핑
4. **시맨틱 처리 파이프라인**: 대화 이벤트에서 시맨틱 메모리로 변환을 위한 90초 대기
5. **동적 세션 관리**: 유연한 세션 처리를 위해 memory.context.session_id 사용

**학습 내용:**

- 여러 연구 세션에 걸친 영구적 AgentCore 메모리 생성
- 시간에 따른 누적 연구 지식 구축
- 연구 이력에 걸친 시맨틱 검색 구현
- 연구 발전 및 전문성 개발 추적
- 크로스 세션 메모리 지속성 및 검색 테스트

## 사전 요구사항

- Python 3.10+
- 적절한 권한이 있는 AWS 계정
- AgentCore 메모리 권한이 있는 AWS IAM 역할:
  - `bedrock-agentcore:CreateMemory`
  - `bedrock-agentcore:CreateEvent`
  - `bedrock-agentcore:ListEvents`
  - `bedrock-agentcore:RetrieveMemories`
- Amazon Bedrock 모델 접근 권한

## 1단계: 의존성 설치 및 설정

In [ ]:
# 시맨틱 전략 툴킷을 포함한 필요한 라이브러리 설치
%pip install llama-index-memory-bedrock-agentcore llama-index-llms-bedrock-converse boto3 bedrock-agentcore-starter-toolkit

In [ ]:
# 필요한 컴포넌트 임포트
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from bedrock_agentcore.memory.session import MemorySessionManager
from bedrock_agentcore_starter_toolkit.operations.memory.models.strategies.semantic import SemanticStrategy
from llama_index.memory.bedrock_agentcore import AgentCoreMemory, AgentCoreMemoryContext
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.tools import FunctionTool
from datetime import datetime
import os

print("모든 의존성 임포트 완료!")

## 2단계: AgentCore 메모리 구성

장기 연구 지식을 위한 AgentCore 메모리 리소스를 생성하거나 가져옵니다:

In [ ]:
# 장기 지속성을 위한 시맨틱 전략이 적용된 AgentCore 메모리 생성
region = os.getenv('AWS_REGION', 'us-east-1')
memory_manager = MemoryManager(region_name=region)

try:
    # 자동 인사이트 추출을 위한 시맨틱 전략으로 메모리 생성
    memory = memory_manager.get_or_create_memory(
        name=f'AcademicResearchSemantic_{int(datetime.now().timestamp())}',
        strategies=[SemanticStrategy(name="researchLongTermMemory")],
        event_expiry_days=365  # 연구 기록을 위한 12개월 보존
    )
    memory_id = memory.get('id')
    print(f"시맨틱 메모리 생성 완료: {memory_id}")
    print(f"   상태: {memory.get('status')}")
    print(f"   전략: {[s.get('name') if isinstance(s, dict) else str(s) for s in memory.get('strategies', [])]}")
    
    # 메모리가 ACTIVE 상태가 될 때까지 대기
    if memory.get('status') != 'ACTIVE':
        print(f"\n메모리가 ACTIVE 상태가 될 때까지 대기 중 (현재 {memory.get('status')})...")
        import time
        max_wait = 300  # 최대 5분
        waited = 0
        while waited < max_wait:
            time.sleep(10)
            waited += 10
            # 상태 확인
            current_memory = memory_manager.get_memory(memory_id)
            status = current_memory.get('status')
            print(f"   [{waited}초] 상태: {status}")
            if status == 'ACTIVE':
                print(f"메모리가 ACTIVE 상태입니다! ({waited}초 소요)")
                break
        else:
            print(f"메모리가 {max_wait}초 후에도 ACTIVE 상태가 아닙니다. 계속 진행합니다...")
    
except Exception as e:
    print(f"메모리 생성 오류: {e}")
    memory_id = "your-memory-id-here"  # 기존 메모리 ID로 교체하세요

## 3단계: 연구 도구 구현

학술 연구 작업을 위한 전문 도구를 정의합니다:

In [ ]:
def save_paper_summary(title: str, authors: str, key_findings: str) -> str:
    """Save a research paper summary with title, authors, and key findings"""
    print(f"논문 저장 완료: {title} by {authors}")
    return f"Successfully saved paper summary for '{title}'"

def track_research_topic(topic: str, status: str) -> str:
    """Track research topic progress with current status"""
    print(f"연구 주제 추적 중: {topic} (상태: {status})")
    return f"Now tracking research topic: {topic} with status {status}"

def save_research_finding(finding: str, confidence: str) -> str:
    """Save a research finding with confidence level"""
    print(f"연구 결과 저장 완료 (신뢰도: {confidence})")
    return f"Saved research finding with {confidence} confidence level"

def update_research_status(topic: str, new_status: str, notes: str) -> str:
    """Update research topic status with notes"""
    print(f"{topic} 상태 업데이트: {new_status}")
    return f"Updated research status for {topic}"

def log_research_milestone(period: str, milestone: str, details: str) -> str:
    """Log a research milestone with period and detailed progress"""
    print(f"{period} 마일스톤: {milestone}")
    return f"Logged milestone for {period}: {milestone} - {details}"

def track_research_metrics(metric_type: str, value: str, source: str, period: str) -> str:
    """Track specific research metrics with source and timeline"""
    print(f"{period}: {metric_type} = {value} (출처: {source})")
    return f"Tracked {metric_type}: {value} from {source} in {period}"

def save_research_insight(insight: str, period: str, connections: str) -> str:
    """Save research insights with connections to previous work"""
    print(f"{period} 인사이트: {insight[:50]}...")
    return f"Saved {period} insight with connections: {connections}"

# 에이전트용 도구 객체 생성
research_tools = [
    FunctionTool.from_defaults(fn=save_paper_summary),
    FunctionTool.from_defaults(fn=track_research_topic),
    FunctionTool.from_defaults(fn=save_research_finding),
    FunctionTool.from_defaults(fn=update_research_status),
    FunctionTool.from_defaults(fn=log_research_milestone),
    FunctionTool.from_defaults(fn=track_research_metrics),
    FunctionTool.from_defaults(fn=save_research_insight)
]

print("연구 도구 생성 완료!")

## 3b단계: 메모리 검색 도구 추가

에이전트가 장기 메모리를 검색할 수 있는 도구를 생성합니다:

In [ ]:
def create_memory_retrieval_tool(memory_id: str, actor_id: str, region: str):
    """에이전트가 자체 장기 메모리를 검색할 수 있는 도구 생성"""
    
    def search_long_term_memory(query: str) -> str:
        """Search long-term memory for relevant research information.
        
        Use this tool when you need to recall:
        - Previous research papers and findings
        - Research topics and their status
        - Metrics and insights from past work
        - Research milestones and progress
        
        Args:
            query: Search query describing what information you need
        
        Returns:
            Relevant information from long-term memory
        """
        try:
            from bedrock_agentcore.memory.session import MemorySessionManager
            
            # 세션 매니저 생성
            session_manager = MemorySessionManager(
                memory_id=memory_id,
                region_name=region
            )
            
            # 시맨틱 전략 네임스페이스에서 장기 메모리 검색
            results = session_manager.search_long_term_memories(
                query=query,
                namespace_prefix="/strategies/",  # 시맨틱 전략 네임스페이스에서 검색
                top_k=5,
                max_results=10
            )
            
            if not results:
                return "No relevant information found in long-term memory. This might be new information or the memory extraction may still be processing."
            
            # 에이전트를 위한 결과 포매팅
            output = "장기 메모리에서 검색된 정보:\\n\\n"
            for i, result in enumerate(results, 1):
                # MemoryRecord 객체 - content 속성 접근
                content = getattr(result, 'content', str(result))
                # 매우 긴 콘텐츠 잘라내기
                if len(content) > 300:
                    content = content[:300] + "..."
                output += f"{i}. {content}\\n\\n"
            
            return output
            
        except Exception as e:
            return f"메모리 검색 오류: {str(e)}. 과거 컨텍스트 없이 진행합니다."
    
    return FunctionTool.from_defaults(fn=search_long_term_memory)

# 메모리 검색 도구 생성
memory_search_tool = create_memory_retrieval_tool(memory_id, "academic-researcher", region)

# 도구 목록에 메모리 검색 추가
research_tools_with_memory = research_tools + [memory_search_tool]

print(f"메모리 검색 도구 생성 완료! 총 도구 수: {len(research_tools_with_memory)}")
print("   사용 네임스페이스: /strategies/ (시맨틱 전략 호환)")

## 3c단계: 메모리 구성 확인

시맨틱 전략이 올바르게 구성되었는지 확인합니다:

In [ ]:
# 메모리 구성 확인
memory_info = memory_manager.get_memory(memory_id)
print(f"전략: {memory_info.get('strategies')}")
print(f"상태: {memory_info.get('status')}")
print(f"이름: {memory_info.get('name')}")

# 전략 세부 정보 표시
strategies = memory_info.get('strategies', [])
for strategy in strategies:
    print(f"\n전략 세부 정보:")
    print(f"  이름: {strategy.get('name')}")
    print(f"  유형: {strategy.get('type')}")
    print(f"  상태: {strategy.get('status')}")
    print(f"  ID: {strategy.get('strategyId')}")

## 4단계: 멀티 세션 에이전트 구현

다양한 연구 세션을 시뮬레이션하기 위한 헬퍼 함수를 생성합니다:

In [ ]:
# 장기 메모리 구성 (크로스 세션)
MODEL_ID = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
RESEARCHER_ID = "academic-researcher"  # 모든 세션에서 동일한 연구자

def create_research_session(session_name: str):
    """장기 메모리 지속성을 가진 새 연구 세션 생성"""
    context = AgentCoreMemoryContext(
        actor_id=RESEARCHER_ID,         # 동일한 연구자
        memory_id=memory_id,         # 동일한 메모리 저장소 (장기 메모리 활성화)
        session_id=f"research-{session_name}", # 기간별 다른 세션
        namespace="/academic-research/"
    )
    
    memory = AgentCoreMemory(context=context)
    llm = BedrockConverse(model=MODEL_ID)
    agent = FunctionAgent(
        tools=research_tools_with_memory,  # 메모리 검색 기능이 있는 도구 사용
        llm=llm, 
        verbose=True,  # 메모리 검색 시 확인을 위한 상세 모드 활성화
        system_prompt="""You are a senior research assistant with access to long-term memory.
        
CRITICAL: When asked about previous research, papers, findings, or historical information, 
you MUST use the search_long_term_memory tool FIRST before responding.

For example:
- "What research am I working on?" → Use search_long_term_memory("research topics")
- "What papers have I reviewed?" → Use search_long_term_memory("papers authors")
- "What findings do I have?" → Use search_long_term_memory("research findings")

Always provide conclusive, complete responses without asking follow-up questions.\n
Execute all requested actions immediately and completely. Provide detailed, professional responses."""
    )
    
    return agent, memory

print("멀티 세션 학술 연구 어시스턴트 설정 완료!")


## 5단계: 1주차 연구 세션 - 기초 구축

첫 번째 연구 세션을 시작하고 기초 지식을 확립합니다:

In [ ]:
# === 1주차 연구 세션 ===
print("=== 1주차: 기초 연구 ===")

agent_week1, memory_week1 = create_research_session("week1")

# 연구 기초 확립
# (MIT의 Sarah Smith 박사로서 '헬스케어에서의 머신러닝 응용' 연구 시작, 문헌 검토 상태, 연말까지 체계적 리뷰 출판 목표)
response = await agent_week1.run(
    "I'm Dr. Sarah Smith from MIT starting comprehensive research on 'Machine Learning in Healthcare Applications'. "
    "Track this with status 'Literature Review'. My goal is to publish a systematic review by year-end.",
    memory=memory_week1
)

print("1주차 기초:")
print(response)

In [ ]:
# 상세 지표가 포함된 기초 논문 추가
# (Zhang et al.(2023)의 '의료 영상 분석을 위한 딥러닝' - CNN이 흉부 X선 진단에서 95.2% 정확도, 방사선 전문의 대비 12% 향상)
response = await agent_week1.run(
    "Save paper: 'Deep Learning for Medical Image Analysis' by Zhang et al (2023). "
    "Key findings: CNNs achieve 95.2% accuracy in chest X-ray diagnosis, 12% improvement over radiologists, "
    "trained on 100,000 images, 0.03 false positive rate.",
    memory=memory_week1
)
print("1주차 논문 1:", response)

# (Johnson et al.(2023)의 '의료 NLP에서의 트랜스포머' - BERT가 임상 노트 분류에서 89.1% F1-스코어, 희귀 질환에서 70% 미만 정확도)
response = await agent_week1.run(
    "Save paper: 'Transformers in Medical NLP' by Johnson et al (2023). "
    "Key findings: BERT achieves 89.1% F1-score in clinical note classification, "
    "struggles with rare diseases (<70% accuracy), excels at symptom extraction (94% precision).",
    memory=memory_week1
)
print("1주차 논문 2:", response)
# 정확도 지표를 명시적으로 추적
await agent_week1.run(
    "Track research metrics: metric_type 'CNN Accuracy', value '95.2%', source 'Zhang et al 2023', period 'Week 1'.",
    memory=memory_week1
)
await agent_week1.run(
    "Track research metrics: metric_type 'Radiologist Improvement', value '12%', source 'Zhang et al 2023', period 'Week 1'.",
    memory=memory_week1
)


In [ ]:
# 시맨틱 메모리 처리를 위한 대기 시간
import asyncio
print("\n시맨틱 메모리 추출 및 인덱싱 대기 중...")
print("   (AgentCore가 백그라운드에서 대화 이벤트를 처리합니다)")
await asyncio.sleep(90)  # 메모리 추출을 위한 대기 시간 증가
print("메모리 처리 완료 - 이제 메모리를 검색할 수 있습니다")


## 6단계: 2주차 연구 세션 - 크로스 세션 메모리 테스트

장기 메모리 검색을 테스트하고 새로운 연구를 추가합니다:

In [ ]:
# === 2주차 연구 세션 ===
print("\n=== 2주차: 확장 (새 세션) ===")

agent_week2, memory_week2 = create_research_session("week2")

# 크로스 세션 메모리 회상 테스트
# (어떤 연구를 하고 있는지, 발견한 구체적 정확도 지표, 주요 저자 질문)
response = await agent_week2.run(
    "What research am I working on? What specific accuracy metrics have I found so far? Who are the key authors?",
    memory=memory_week2
)

print("2주차 메모리 테스트:")
print(response)
print("\n예상 결과: ML in Healthcare, Zhang 95.2%, Johnson 89.1% F1-스코어")

In [ ]:
# 이전 지식을 기반으로 새로운 연구 추가
# (Brown et al.(2023)의 '헬스케어에서의 연합 학습' - 15개 병원에서 87.3% 정확도, 병원 협력 시 희귀 질환 탐지 23% 향상)
response = await agent_week2.run(
    "Save paper: 'Federated Learning in Healthcare' by Brown et al (2023). "
    "Key findings: Privacy-preserving ML enables multi-hospital collaboration, 87.3% accuracy across 15 hospitals, "
    "23% improvement in rare disease detection when hospitals collaborate.",
    memory=memory_week2
)
print("2주차 새 논문:", response)

# 세션 간 비교 분석 테스트
# (Zhang의 CNN vs Johnson의 BERT vs Brown의 연합 학습 정확도 결과 비교 - 어떤 것이 최고이며 어떤 상황에서인지)
response = await agent_week2.run(
    "Compare the accuracy results: Zhang's CNNs vs Johnson's BERT vs Brown's federated learning. "
    "Which performs best and in what contexts?",
    memory=memory_week2
)
print("2주차 비교 분석:")
print(response)
print("\n예상 결과: Zhang 95.2% (영상), Johnson 89.1% (NLP), Brown 87.3% (연합)")

## 7단계: 3주차 연구 세션 - 분석 단계

연구를 진행하고 상세한 크로스 세션 회상을 테스트합니다:

In [ ]:
# === 3주차 연구 세션 ===
print("\n=== 3주차: 분석 단계 ===")

agent_week3, memory_week3 = create_research_session("week3")

# 연구 상태 업데이트
# ('헬스케어에서의 머신러닝 응용' 연구 상태를 '분석 단계'로 업데이트 - 3개 핵심 논문 검토 완료, 성능 패턴 식별)
response = await agent_week3.run(
    "Update my 'Machine Learning in Healthcare Applications' research status to 'Analysis Phase' "
    "with notes: 'Reviewed 3 key papers, identified performance patterns: imaging>NLP>federated learning'.",
    memory=memory_week3
)
print("3주차 상태 업데이트:", response)

# 상세 크로스 세션 회상 테스트
# (헬스케어에서 영상 작업이 가장 높은 ML 성능을 보인다는 주장에 대한 근거 - 구체적 수치와 저자 포함)
response = await agent_week3.run(
    "What evidence do I have for the claim that imaging tasks show highest ML performance in healthcare? "
    "Include specific numbers and authors.",
    memory=memory_week3
)
print("3주차 근거 질의:")
print(response)
print("\n예상 결과: Zhang et al CNN 95.2% vs Johnson BERT 89.1% vs Brown 연합 87.3%")

## 8단계: 1개월차 연구 세션 - 종합 단계

포괄적인 지식 종합 및 연구 통합을 테스트합니다:

In [ ]:
# === 1개월차 연구 세션 ===
print("\n=== 1개월차: 종합 단계 ===")

agent_month1, memory_month1 = create_research_session("month1")

# 연구 상태를 종합 단계로 업데이트
response = await agent_month1.run(
    "Update my 'Machine Learning in Healthcare Applications' research status to 'Synthesis Phase' "
    "with notes: 'Completed 3-week literature review, ready to synthesize findings into coherent framework'.",
    memory=memory_month1
)
print("1개월차 상태 업데이트:", response)

# 모든 주차에 걸친 종합 합성 테스트
# (지금까지의 모든 연구를 기반으로 헬스케어에서의 ML 접근법 전체 성능 순위 - 모든 구체적 지표 포함)
response = await agent_month1.run(
    "Based on all my research so far, what is the overall performance ranking of ML approaches in healthcare? "
    "Include all specific metrics and create a comprehensive comparison.",
    memory=memory_month1
)
print("1개월차 종합 합성:")
print(response)
print("\n예상 결과: Zhang 95.2% > Johnson 89.1% > Brown 87.3%, 도메인 분석 포함 순위")

## 9단계: 2개월차 연구 세션 - 집필 단계

포괄적인 회상 및 시맨틱 검색 기능을 테스트합니다:

In [ ]:
# === 2개월차 연구 세션 ===
print("\n=== 2개월차: 집필 단계 ===")

agent_month2, memory_month2 = create_research_session("month2")

# 집필을 위한 종합 회상 테스트
# (체계적 리뷰 논문을 작성 중이므로, 검토한 모든 논문과 정확한 정확도 지표 필요 - 결과 표에 사용)
response = await agent_month2.run(
    "I'm writing my systematic review paper. What are ALL the papers I've reviewed with their exact accuracy metrics? "
    "I need this for my results table.",
    memory=memory_month2
)
print("2개월차 종합 회상:")
print(response)
print("\n예상 결과: Zhang 95.2%, Johnson 89.1%, Brown 87.3% 전체 세부 정보 포함")

In [ ]:
# 연구 이력에 걸친 시맨틱 검색 테스트
# (연구에서 희귀 질환 탐지에 대해 알고 있는 것 질문 - 어떤 논문과 구체적 결과)
response = await agent_month2.run(
    "What do I know about rare disease detection in my research? Which papers and what specific results?",
    memory=memory_month2
)
print("2개월차 시맨틱 검색:")
print(response)
print("\n예상 결과: Johnson 희귀 질환 70% 미만, Brown 협력 시 23% 향상")

## 10단계: 3개월차 연구 세션 - 연구비 제안서 시나리오

축적된 지식의 실용적 응용을 테스트합니다:

In [ ]:
# === 3개월차 연구 세션 ===
print("\n=== 3개월차: 연구비 제안서 ===")

agent_month3, memory_month3 = create_research_session("month3")

# 연구비 제안서 근거 수집
# ($2M 자금의 NIH 연구비 제안서를 작성 중 - 헬스케어에서의 ML 효과에 대한 근거 필요, 구체적 수치/저자/연도/샘플 크기 포함)
response = await agent_month3.run(
    "I'm writing an NIH grant proposal for $2M funding. What evidence can I cite about ML effectiveness in healthcare? "
    "I need specific numbers, authors, years, and sample sizes.",
    memory=memory_month3
)
print("3개월차 연구비 근거:")
print(response)
print("\n예상 결과: Zhang 95.2% (10만 이미지), Johnson 89.1%, Brown 87.3% (15개 병원) 포함 종합 인용")

In [ ]:
# 상세 마일스톤이 포함된 연구 발전 추적 테스트
# (1주차부터 현재까지의 상세한 연구 발전 타임라인 - 각 기간의 구체적 마일스톤, 지표, 인사이트 및 연구 질문의 진화)
response = await agent_month3.run(
    "Provide a detailed timeline of my research evolution from Week 1 to now. What specific milestones, "
    "metrics, and insights did I achieve each period? How did my research questions evolve?",
    memory=memory_month3
)
print("3개월차 연구 발전:")
print(response)
print("\n예상 결과: 구체적 마일스톤, 지표(95.2%, 89.1%, 87.3%), 인사이트가 포함된 주차별 진행 상황")

## 11단계: 최종 포트폴리오 평가

장기 메모리 기능에 대한 종합 테스트:

In [ ]:
# 최종 종합 포트폴리오 질의
# (전체 연구 포트폴리오 - 진행 중인 모든 주제, 지표가 포함된 모든 논문, 모든 결과, 각 프로젝트의 현재 상태 및 상호 연결 방식)
response = await agent_month3.run(
    "Provide my complete research portfolio: all topics I'm working on, all papers with metrics, "
    "all findings, current status of each project, and how they interconnect.",
    memory=memory_month3
)
print("전체 연구 포트폴리오:")
print(response)
print("\n예상 결과: 모든 지표 포함 전체 연구 이력, ML 헬스케어 주제 간 연결")

## 자동화된 테스트 검증
메모리 통합이 올바르게 작동하는지 검증하기 위해 다음 셀들을 실행하세요:

In [ ]:
# 검증 함수 인라인 정의
class TestValidator:
    def __init__(self):
        self.results = {}
    
    def validate_memory_recall(self, response):
        """에이전트가 세션 초반 정보를 회상할 수 있는지 확인"""
        has_content = len(response) > 50
        has_memory_indicators = any(word in response.lower() for word in 
            ['earlier', 'mentioned', 'discussed', 'previously', 'you', 'we', 'our'])
        return "통과" if (has_content and has_memory_indicators) else "실패"
    
    def validate_session_memory(self, response):
        """에이전트가 세션 내에서 컨텍스트를 유지하는지 확인"""
        has_memory_content = len(response) > 100 and any(word in response.lower() for word in 
            ['previous', 'earlier', 'mentioned', 'discussed', 'before', 'already'])
        return "통과" if has_memory_content else "실패"
    
    def validate_cross_reference(self, response):
        """에이전트가 현재 쿼리를 이전 컨텍스트와 연결할 수 있는지 확인"""
        connecting_words = ['relate', 'connection', 'previous', 'earlier', 'discussed', 
                           'mentioned', 'context', 'based on', 'as we', 'as i']
        has_connection = any(word in response.lower() for word in connecting_words)
        has_substance = len(response) > 80
        return "통과" if (has_connection and has_substance) else "실패"
    
    def run_validation_summary(self, test_results):
        print("종합 테스트 검증 요약")
        print("=" * 60)
        
        total_tests = len(test_results)
        passed_tests = sum(1 for result in test_results.values() if "통과" in result)
        pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0
        
        for test_name, result in test_results.items():
            print(f"{test_name}: {result}")
        
        print("=" * 60)
        print(f"전체 통과율: {passed_tests}/{total_tests} ({pass_rate:.1f}%)")
        
        if pass_rate >= 80:
            print("우수: 메모리 통합이 올바르게 작동합니다!")
        elif pass_rate >= 60:
            print("양호: 대부분의 메모리 기능이 작동하며, 일부 조사가 필요합니다")
        else:
            print("주의 필요: 메모리 통합에 중요한 문제가 있습니다")
        
        return pass_rate

validator = TestValidator()
print("검증 함수 로드 완료!")

In [ ]:
# 모든 검증 테스트 실행
test_results = {}

# 테스트 1: 메모리 회상
response1 = await agent_month3.run("What have we discussed so far in this session?", memory=memory_month3)
test_results['메모리 회상'] = validator.validate_memory_recall(str(response1))
print(f"응답 1 길이: {len(str(response1))} 문자")

# 테스트 2: 세션 메모리
response2 = await agent_month3.run("What did we talk about earlier?", memory=memory_month3)
test_results['세션 메모리'] = validator.validate_session_memory(str(response2))
print(f"응답 2 길이: {len(str(response2))} 문자")

# 테스트 3: 교차 참조 기능
response3 = await agent_month3.run("How does this relate to what we discussed before?", memory=memory_month3)
test_results['교차 참조'] = validator.validate_cross_reference(str(response3))
print(f"응답 3 길이: {len(str(response3))} 문자")

# 결과 표시
validator.run_validation_summary(test_results)

## 요약

이 노트북에서 다음을 시연했습니다:

- **장기 메모리 통합**: 크로스 세션 지속성을 위해 LlamaIndex와 AgentCore 메모리 사용

- **누적 지식 구축**: 수주 및 수개월에 걸쳐 연구 지식이 축적됨

- **시맨틱 검색**: 세션에 걸쳐 개념 기반으로 관련 정보를 찾는 어시스턴트

- **연구 발전 추적**: 문헌 검토에서 분석, 집필까지의 자연스러운 진행

- **크로스 세션 합성**: 여러 연구 세션에 걸쳐 결과와 인사이트를 연결

- **실용적 응용**: 연구비 제안서 지원 및 종합 포트폴리오 관리

학술 연구 어시스턴트는 장기 메모리가 어떻게 어시스턴트를 시간이 지남에 따라 더 똑똑해지는 영구적 연구 동반자로 변환하며, 완전한 연구 이력을 유지하고 확장된 연구 프로젝트에 걸쳐 정교한 지식 검색을 가능하게 하는지 보여줍니다.

## 정리

이 노트북에서 사용한 리소스를 정리하기 위해 메모리를 삭제하겠습니다:

**참고**: 메모리를 영구적으로 삭제하려는 경우에만 실행하세요. memory_id 변수에는 이 노트북 앞부분에서 생성한 메모리의 ID가 포함되어 있어야 합니다.

In [ ]:
# AgentCore 메모리 리소스 정리
try:
    from bedrock_agentcore.memory import MemoryClient
    
    client = MemoryClient(region_name=region)
    client.delete_memory(memory_id)
    print(f"메모리 삭제 완료: {memory_id}")
    
except NameError as e:
    print(f"변수가 정의되지 않음: {e}")
    print("노트북을 처음부터 실행하거나 변수를 수동으로 설정하세요:")
    print("# memory_id = 'your-memory-id-here'")
    print("# region = 'us-east-1'")
except Exception as e:
    print(f"메모리 삭제 오류: {e}")